In [4]:
import pandas as pd
import numpy as np

from scipy.stats import chisquare, norm
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.multitest import multipletests

from google.colab import drive

drive.mount('/content/drive')

import glob

matches = glob.glob(
    '/content/drive/MyDrive/**/Project 4.csv',
    recursive=True
)

matches

df = pd.read_csv(matches[0])

df

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,metric,control_n,control_success,treatment_n,treatment_success
0,onboarding_completion,5677,3020,5785,3271
1,trial_start,5677,889,5785,989
2,paid_conversion,5677,576,5785,547
3,D1_retention,5677,2276,5785,2393
4,D7_retention,5677,1063,5785,1041
5,D30_retention,5677,823,5785,869


In [5]:
# Sample Ratio Mismatch check

control_n = df.loc[0, 'control_n']
treatment_n = df.loc[0, 'treatment_n']

observed = np.array([control_n, treatment_n])

total = observed.sum()
expected = np.array([total / 2, total / 2])

chi2_stat, srm_p_value = chisquare(
    f_obs=observed,
    f_exp=expected
)

print(f'Control:   {control_n}')
print(f'Treatment: {treatment_n}')
print(f'Chi-square statistic: {chi2_stat:.4f}')
print(f'SRM p-value: {srm_p_value:.4f}')

if srm_p_value < 0.05:
    print('SRM detected: allocation may be broken.')
else:
    print('No SRM detected: allocation is consistent with 50/50.')

Control:   5677
Treatment: 5785
Chi-square statistic: 1.0176
SRM p-value: 0.3131
No SRM detected: allocation is consistent with 50/50.


In [6]:
alpha = 0.05
results = []

for _, row in df.iterrows():

    metric = row['metric']

    control_n = int(row['control_n'])
    control_success = int(row['control_success'])

    treatment_n = int(row['treatment_n'])
    treatment_success = int(row['treatment_success'])

    control_rate = control_success / control_n
    treatment_rate = treatment_success / treatment_n

    diff = treatment_rate - control_rate
    diff_pp = diff * 100

    relative_uplift = (
        diff / control_rate * 100
        if control_rate > 0 else np.nan
    )

    # Two-proportion z-test
    counts = np.array([
        treatment_success,
        control_success
    ])

    nobs = np.array([
        treatment_n,
        control_n
    ])

    z_stat, p_value = proportions_ztest(
        count=counts,
        nobs=nobs,
        alternative='two-sided'
    )

    # 95% CI for treatment - control difference
    se_diff = np.sqrt(
        treatment_rate * (1 - treatment_rate) / treatment_n
        +
        control_rate * (1 - control_rate) / control_n
    )

    z_crit = norm.ppf(0.975)

    ci_low = diff - z_crit * se_diff
    ci_high = diff + z_crit * se_diff

    results.append({
        'metric': metric,

        'control_rate_pct': control_rate * 100,
        'treatment_rate_pct': treatment_rate * 100,

        'difference_pp': diff_pp,
        'relative_uplift_pct': relative_uplift,

        'ci_low_pp': ci_low * 100,
        'ci_high_pp': ci_high * 100,

        'z_stat': z_stat,
        'p_value': p_value
    })


results_df = pd.DataFrame(results)

primary_metric = 'onboarding_completion'

results_df['adjusted_p_value'] = results_df['p_value']
results_df['significant'] = False

# Primary metric: raw p-value
primary_mask = results_df['metric'] == primary_metric

results_df.loc[
    primary_mask,
    'significant'
] = results_df.loc[
    primary_mask,
    'p_value'
] < alpha


# Secondary metrics: Holm correction
secondary_mask = results_df['metric'] != primary_metric

reject, corrected_p, _, _ = multipletests(
    results_df.loc[secondary_mask, 'p_value'],
    alpha=alpha,
    method='holm'
)

results_df.loc[
    secondary_mask,
    'adjusted_p_value'
] = corrected_p

results_df.loc[
    secondary_mask,
    'significant'
] = reject


results_df.round({
    'control_rate_pct': 2,
    'treatment_rate_pct': 2,
    'difference_pp': 2,
    'relative_uplift_pct': 2,
    'ci_low_pp': 2,
    'ci_high_pp': 2,
    'z_stat': 3,
    'p_value': 4,
    'adjusted_p_value': 4
})

,metric,control_rate_pct,treatment_rate_pct,difference_pp,relative_uplift_pct,ci_low_pp,ci_high_pp,z_stat,p_value,adjusted_p_value,significant
0,onboarding_completion,53.20,56.54,3.35,6.29,1.52,5.17,3.599,0.0003,0.0003,True
1,trial_start,15.66,17.10,1.44,9.17,0.08,2.79,2.077,0.0378,0.1890,False
2,paid_conversion,10.15,9.46,-0.69,-6.81,-1.78,0.40,-1.244,0.2136,0.6606,False
3,D1_retention,40.09,41.37,1.27,3.18,-0.52,3.07,1.388,0.1652,0.6606,False
4,D7_retention,18.72,17.99,-0.73,-3.90,-2.15,0.69,-1.009,0.3129,0.6606,False
5,D30_retention,14.50,15.02,0.52,3.62,-0.77,1.82,0.792,0.4287,0.6606,False


In [7]:
# Sensitivity analysis for paid-conversion rollout guardrail

paid = results_df.loc[
    results_df['metric'] == 'paid_conversion'
].iloc[0]

observed_diff_pp = paid['difference_pp']
ci_low_pp = paid['ci_low_pp']
ci_high_pp = paid['ci_high_pp']

# Hypothetical business guardrails:
# maximum acceptable decrease in paid conversion
thresholds = [-0.25, -0.50, -1.00, -1.50, -2.00]

sensitivity = pd.DataFrame({
    'max_allowed_drop_pp': thresholds
})

sensitivity['observed_effect_pp'] = observed_diff_pp
sensitivity['ci_low_pp'] = ci_low_pp
sensitivity['ci_high_pp'] = ci_high_pp

# Safe only if the entire confidence interval
# stays above the business guardrail
sensitivity['guardrail_supported'] = (
    sensitivity['ci_low_pp']
    > sensitivity['max_allowed_drop_pp']
)

sensitivity['decision'] = np.where(
    sensitivity['guardrail_supported'],
    'Safety supported',
    'Insufficient evidence'
)

sensitivity.round(2)

,max_allowed_drop_pp,observed_effect_pp,ci_low_pp,ci_high_pp,guardrail_supported,decision
0,-0.25,-0.69,-1.78,0.4,False,Insufficient evidence
1,-0.50,-0.69,-1.78,0.4,False,Insufficient evidence
2,-1.00,-0.69,-1.78,0.4,False,Insufficient evidence
3,-1.50,-0.69,-1.78,0.4,False,Insufficient evidence
4,-2.00,-0.69,-1.78,0.4,True,Safety supported


In [8]:
# Final A/B test export for Tableau

ab_test_results = results_df.copy()

ab_test_results.to_csv(
    "ab_test_results.csv",
    index=False
)

from google.colab import files
files.download("ab_test_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>